<a href="https://colab.research.google.com/github/garnfelfel/Deep_Learning_Homeworks/blob/main/ctcBeamSearchTransformer_DL_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip cache purge
!pip uninstall numpy
!pip install numpy pyctcdecode jiwer https://github.com/kpu/kenlm/archive/master.zip huggingface_hub

Files removed: 0
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Would remove:
    /usr/local/bin/f2py
    /usr/local/bin/numpy-config
    /usr/local/lib/python3.12/dist-packages/numpy-2.0.2.dist-info/*
    /usr/local/lib/python3.12/dist-packages/numpy.libs/libgfortran-040039e1-0352e75f.so.5.0.0
    /usr/local/lib/python3.12/dist-packages/numpy.libs/libquadmath-96973f99-934c22de.so.0.0.0
    /usr/local/lib/python3.12/dist-packages/numpy.libs/libscipy_openblas64_-99b71e71.so
    /usr/local/lib/python3.12/dist-packages/numpy/*
Proceed (Y/n)? ERROR: Operation cancelled by user
     | 553.6 kB 2.6 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.5/537.5 kB 36.1 MB/s et

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import os
import torch
import numpy as np
prep_dir = "/content/drive/MyDrive/EEG-to-Text/preprocessed_frames_flat"


files = [f for f in os.listdir(prep_dir) if f.endswith(".pt")]
print("Total .pt files:", len(files))
print("Sample names FILES:", files[:15])

chars = set()
words = []
for f in files:
    word = f.split("_")[0].lower()
    chars.update(list(word))
    words.append(word)

chars = sorted(chars)

print("Characters:", chars)
print("Sample words : ", words[:10])
# Dictionaries for mapping
char_to_int = {}
int_to_char = {}
BLANK_TOKEN_INDEX = 0
# Add the CTC blank token
char_to_int['<BLANK>'] = BLANK_TOKEN_INDEX
int_to_char[BLANK_TOKEN_INDEX] = '<BLANK>'

# Define vocabulary

# Add characters starting from index 1
for i, char in enumerate(chars, start=1):
    char_to_int[char] = i
    int_to_char[i] = char

# Total number of classes
NUM_GRAPHEMES = len(char_to_int)


print("Number of graphemes:", NUM_GRAPHEMES)
print("char_to_int :", char_to_int)
print("int_to_char :", int_to_char)
print("Example mapping:", char_to_int['a'], "->", int_to_char[char_to_int['a']])

# Create the Vocabulary JSON File
LABELS_FILE_PATH = '/content/drive/MyDrive/EEG-to-Text/labels.json'
labels = [int_to_char.get(i, '') for i in range(len(int_to_char))]
labels[BLANK_TOKEN_INDEX] = ""  # Blank token must be an empty string
with open(LABELS_FILE_PATH, 'w') as f:
    json.dump(labels, f)
print(f"Created {LABELS_FILE_PATH} with {len(labels)} graphemes.")

Total .pt files: 995
Sample names FILES: ['helft_sub-06.pt', 'helft_sub-04.pt', 'helft_sub-10.pt', 'helft_sub-01.pt', 'helft_sub-02.pt', 'helft_sub-03.pt', 'helft_sub-07.pt', 'hierop_sub-01.pt', 'helft_sub-05.pt', 'hierop_sub-02.pt', 'hierop_sub-03.pt', 'hierop_sub-04.pt', 'hierop_sub-05.pt', 'hierop_sub-09.pt', 'hierop_sub-07.pt']
Characters: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'z']
Sample words :  ['helft', 'helft', 'helft', 'helft', 'helft', 'helft', 'helft', 'hierop', 'helft', 'hierop']
Number of graphemes: 35
char_to_int : {'<BLANK>': 0, '0': 1, '1': 2, '2': 3, '3': 4, '4': 5, '5': 6, '6': 7, '7': 8, '8': 9, '9': 10, '`': 11, 'a': 12, 'b': 13, 'c': 14, 'd': 15, 'e': 16, 'f': 17, 'g': 18, 'h': 19, 'i': 20, 'j': 21, 'k': 22, 'l': 23, 'm': 24, 'n': 25, 'o': 26, 'p': 27, 'r': 28, 's': 29, 't': 30, 'u': 31, 'v': 32, 'w': 33, 'z': 34}
int_to_char : {0: '<BLAN

In [ ]:
for f in files[:5]:
    path = os.path.join(prep_dir, f)
    x = torch.load(path)  # [frames, window, channels] BEFORE flattening in Dataset (if you haven’t flattened when saving)
    print("\nFile:", f)
    print("  shape:", x.shape)
    print("  mean:", x.mean().item())
    print("  std :", x.std().item())
    print("  min :", x.min().item())
    print("  max :", x.max().item())
    print("  any NaN:", torch.isnan(x).any().item())



File: helft_sub-06.pt
  shape: torch.Size([200, 51, 127])
  mean: -0.00474343029782176
  std : 0.9972751140594482
  min : -3.4382150173187256
  max : 3.2771613597869873
  any NaN: False

File: helft_sub-04.pt
  shape: torch.Size([200, 51, 115])
  mean: 0.0021864413283765316
  std : 0.9998716115951538
  min : -4.983809947967529
  max : 3.4303910732269287
  any NaN: False

File: helft_sub-10.pt
  shape: torch.Size([201, 51, 122])
  mean: 0.004565604496747255
  std : 0.9993846416473389
  min : -5.355626583099365
  max : 4.974572658538818
  any NaN: False

File: helft_sub-01.pt
  shape: torch.Size([200, 51, 127])
  mean: 0.009969652630388737
  std : 1.0009251832962036
  min : -3.3841865062713623
  max : 4.234097480773926
  any NaN: False

File: helft_sub-02.pt
  shape: torch.Size([200, 51, 127])
  mean: 0.00974585022777319
  std : 0.9982383847236633
  min : -3.343153238296509
  max : 4.903892993927002
  any NaN: False


In [ ]:
import torch.nn.utils.rnn as rnn_utils

def ctc_collate_fn(batch):
    """
    Processes a list of (sequence, target) tuples
    and turns them into concatenated batches batches.
    """
    sequences = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    print("sequences:", sequences)
    print("targets:", targets)
    seq_lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    padded_seqs = rnn_utils.pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0.0
    )
    # Pad the targets (batch_first=True for loss)
    # Output shape: [BatchSize, MaxTargetLen]
    targets_concat = torch.cat(targets, dim=0)
    return padded_seqs, targets_concat, seq_lengths, target_lengths

In [ ]:
import math
import kenlm

lm_path = "/content/drive/MyDrive/EEG-to-Text/wiki_nl_token.arpa.bin"
lm = kenlm.Model(lm_path)


#  read the wiki_nl_token.arpa.bin : create the unigram file that should be inputed to the ctc function and filter out any unigrams that have characters not within the characters of my language char_to_int : {'<BLANK>': 0, '0': 1, '1': 2, '2': 3, '3': 4, '4': 5, '5': 6, '6': 7, '7': 8, '8': 9, '9': 10, '`': 11, 'a': 12, 'b': 13, 'c': 14, 'd': 15, 'e': 16, 'f': 17, 'g': 18, 'h': 19, 'i': 20, 'j': 21, 'k': 22, 'l': 23, 'm': 24, 'n': 25, 'o': 26, 'p': 27, 'r': 28, 's': 29, 't': 30, 'u': 31, 'v': 32, 'w': 33, 'z': 34}
int_to_char : {0: '<BLANK>', 1: '0', 2: '1', 3: '2', 4: '3', 5: '4', 6: '5', 7: '6', 8: '7', 9: '8', 10: '9', 11: '`', 12: 'a', 13: 'b', 14: 'c', 15: 'd', 16: 'e', 17: 'f', 18: 'g', 19: 'h', 20: 'i', 21: 'j', 22: 'k', 23: 'l', 24: 'm', 25: 'n', 26: 'o', 27: 'p', 28: 'r', 29: 's', 30: 't', 31: 'u', 32: 'v', 33: 'w', 34: 'z'}


In [ ]:
import torch
import os
from torch.utils.data import Dataset
import torch.nn.functional as F

class EEGWordDataset_MultiSubject(Dataset):
    """
    Loads data from multiple subjects by padding the channel dimension.
    """
    def __init__(self, data_dir, char_map, max_channels, len_data):
        self.data_dir = data_dir
        self.char_map = char_map
        self.max_channels = max_channels # Max channels found in your dataset

        self.samples = []
        for f in os.listdir(data_dir):
            if f.endswith('.pt'):
                word = f.split('_')[0]
                self.samples.append((os.path.join(self.data_dir, f), word))
        self.samples = self.samples[:len_data]
        print(f"Loaded {len(self.samples)} samples from {data_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, word = self.samples[idx]

        # Load the 3D tensor
        # Shape: [num_frames, window_size, num_channels]
        x_tensor = torch.load(file_path)

        num_frames, window_size, num_channels = x_tensor.shape

        # Pad the channel dimension if necessary
        if num_channels < self.max_channels:
            # Calculate padding needed
            pad_width = self.max_channels - num_channels

            # Pad LAST dimension (channels)
            # (pad_left, pad_right, pad_top, pad_bottom, pad_front, pad_back)
            # We only want to pad the last dim: (0, pad_width)
            x_tensor = F.pad(x_tensor, (0, pad_width), "constant", 0)

        # Flatten features
        # Shape: [num_frames, window_size * self.max_channels]
        x_tensor = x_tensor.flatten(start_dim=1)

        # Encode the target word
        target = [self.char_map[char] for char in word.lower() if char in self.char_map]
        y_tensor = torch.tensor(target, dtype=torch.long)

        return x_tensor, y_tensor

In [ ]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1) # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)  # (max_len, d_model)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        seq_len = x.size(1)
        x = x + self.pe[:seq_len].unsqueeze(0)  # (1, seq, d_model)
        return self.dropout(x)


class BCIToTextModel(nn.Module):
    def __init__(self,
                 input_feat_dim: int,
                 num_graphemes: int,
                 nhead: int = 4,
                 d_model: int = 256,
                 num_encoder_layers: int = 3,
                 dim_feedforward: int = 1024,
                 dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model

        # 1) Project input features to model dimension
        self.input_projection = nn.Linear(input_feat_dim, d_model)

        # 2) Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # 3) Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_encoder_layers,
        )

        # 4) Output projection to grapheme logits
        self.fc_out = nn.Linear(d_model, num_graphemes)

        # Initialize output bias conservatively (no special blank hack)
        '''with torch.no_grad():
            self.fc_out.bias[BLANK_TOKEN_INDEX] -= 8.0'''

        # LogSoftmax for CTC
        self.log_softmax = nn.LogSoftmax(dim=2)

    def forward(self, src, src_key_padding_mask=None):
        """
        src: (B, S, F)
        src_key_padding_mask: (B, S) with True at PAD positions
        """
        assert src.dim() == 3, f"src must be 3D (B,S,F). got {src.dim()}D"

        if src_key_padding_mask is not None:
            assert src_key_padding_mask.dim() == 2, "mask must be 2D (B,S)"
            assert src.shape[0] == src_key_padding_mask.shape[0], (
                f"Batch dim mismatch: src.shape[0]={src.shape[0]} vs "
                f"mask.shape[0]={src_key_padding_mask.shape[0]}"
            )

        # Project to d_model
        print(src)
        src = self.input_projection(src) * math.sqrt(self.d_model)  # (B,S,d_model)
        print(src)
        # Add positional encoding
        src = self.pos_encoder(src)                                # (B,S,d_model)
        print(src)
        # Transformer encoder
        memory = self.transformer_encoder(
            src,
            src_key_padding_mask=src_key_padding_mask,
        )                                                           # (B,S,d_model)

        # Project to logits
        logits = self.fc_out(memory)                               # (B,S,C)

        # Log softmax over classes for CTC
        return self.log_softmax(logits)                            # (B,S,C)


In [ ]:

#  read the wiki_nl_token.arpa.bin : create the unigram file that should be inputed to the ctc function and filter out any unigrams that have characters not within the characters of my language char_to_int : {'<BLANK>': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '0': 27, '1': 28, '2': 29, '3': 30, '4': 31, '5': 32, '6': 33, '7': 34, '8': 35, '9': 36, "'": 37, ' ': 38}


In [ ]:
from pyctcdecode import build_ctcdecoder
# Load the CTC Beam Search Decoder
try:
    decoder = build_ctcdecoder(
        labels,
        kenlm_model_path=lm_path,
        alpha=1.0,  # How much to trust the LM
        beta=0.5,   # How much to reward word length
    )
    print("Successfully loaded CTC Beam Search Decoder.")
except Exception as e:
    print(f"Error loading Decoder: {e}")

Successfully loaded CTC Beam Search Decoder.


In [ ]:
def decode_target_batch(target_ids, int_to_char, pad_index=0):
    """Convert padded target indices to strings."""
    decoded_batch = []
    for seq in target_ids:
        chars = [int_to_char.get(idx.item(), '?') for idx in seq                if idx.item() != pad_index]
        decoded_batch.append("".join(chars))
    print("decoded_batch: ", decoded_batch)
    return decoded_batch


def greedy_decode_batch(log_probs, int_to_char, blank_index):
    """
    log_probs must be (T, B, C).
    Greedy CTC decoding with collapsing repeats and removing blanks.
    """
    T, B, C = log_probs.shape
    best_path = torch.argmax(log_probs, dim=2)  # (T, B)
    print("best_path: ", best_path)
    decoded_batch = []

    for b in range(B):
        seq = best_path[:, b]
        decoded_word = []
        last = -1
        for idx in seq:
            idx = idx.item()
            if idx == last:
                continue
            if idx == blank_index:
                last = idx
                continue
            decoded_word.append(int_to_char.get(idx, '?'))
            last = idx
        decoded_batch.append("".join(decoded_word))

    return decoded_batch


def beam_search_decode_batch(log_probs, decoder, expect_log_probs=True):
    """
    LM-based CTC decoding.
    decoder.decode expects shape (T, C) per sample.
    """

    T, B, C = log_probs.shape
    #print("T:", T)
    #print("B:", B)
    #print("C:", C)
    decoded_batch = []
    for b in range(B):
        lp = log_probs[:, b, :]
        # print("lp: ", lp)
        # LM usually expects probabilities
        if expect_log_probs:
            t_probs = lp.exp()
            #print("t_probs: ", t_probs)
            cpu_probs = t_probs.cpu()
            #print("cpu_probs: ", cpu_probs)
            probs = cpu_probs.numpy() # the issue is here
            #print("probs: ", probs)
        else:
            probs = lp.cpu().numpy()
            #print("probs: ", probs)

        text = decoder.decode(probs)
        #print("text: ", text)
        decoded_batch.append(text)
        #print("decoded_batch: ", decoded_batch)

    return decoded_batch


In [ ]:
from torch.utils.data import DataLoader


def trainModel(N_TRAIN,N_VAL, d_model=512,num_encoder_layers = 6,dim_feedforward = 2048, len_dataset=995):
  percentage_str = f"{round(N_TRAIN*100/995)}"
  print(f"\n---------------- for {percentage_str}% : N_train = {N_TRAIN}, N_val = {N_VAL}")

  # --- Datasets ---
  '''train_dataset = EEGWordDataset_MultiSubject(
      data_dir=TRAIN_DIR,
      char_map=char_to_int,
      max_channels=MAX_CHANNELS,
      len_data=N_TRAIN
  )
  val_dataset = EEGWordDataset_MultiSubject(
      data_dir=VAL_DIR,
      char_map=char_to_int,
      max_channels=MAX_CHANNELS,
      len_data=N_VAL
  )'''

  dataset = EEGWordDataset_MultiSubject(
      data_dir=TRAIN_DIR,
      char_map=char_to_int,
      max_channels=MAX_CHANNELS,
      len_data=len_ds
  )
  train_dataset, val_dataset = torch.utils.data.random_split(dataset, [N_TRAIN, len(dataset)-N_TRAIN])
  train_loader = DataLoader(
      train_dataset,
      batch_size=BATCH_SIZE,
      shuffle=True,
      collate_fn=ctc_collate_fn,
      num_workers=0
  )
  val_loader = DataLoader(
      val_dataset,
      batch_size=BATCH_SIZE,
      shuffle=False,
      collate_fn=ctc_collate_fn,
      num_workers=0
  )

  # --- Feature dim discovery ---
  sample_seq, _ = train_dataset[0]
  HFA_FEATURE_DIM = sample_seq.shape[1]  # [SeqLen, FeatureDim]
  print("HFA_FEATURE_DIM:", HFA_FEATURE_DIM)
  # --- Model / loss / optimizer ---
  model = BCIToTextModel(
      input_feat_dim=HFA_FEATURE_DIM,
      num_graphemes=NUM_GRAPHEMES,
      nhead=8,
      d_model=d_model,
      num_encoder_layers=num_encoder_layers,
      dim_feedforward=dim_feedforward
  ).to(DEVICE)

  # make sure model returns log-probs in shape [SeqLen, Batch, Classes] for CTCLoss
  ctc_loss = nn.CTCLoss(blank=BLANK_TOKEN_INDEX)
  optimizer = optim.Adam(model.parameters(), lr=1e-3)

  # --- Training ---
  model.train()

  for epoch in range(NUM_EPOCHS):
      epoch_loss = 0.0
      for batch_idx, batch in enumerate(train_loader):  # renamed idx to avoid shadowing
          padded_seqs, padded_targets, seq_lengths, target_lengths = [d.to(DEVICE) for d in batch]
          padded_seqs = padded_seqs[:, ::2, :]
          seq_lengths = (seq_lengths + 1) // 2

          max_seq_len = padded_seqs.shape[1]
          padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                          .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))
          # padding mask: SeqLen is dim 1 (Batch, SeqLen, FeatureDim)
          max_seq_len = padded_seqs.shape[1]
          padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                          .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))
          #print("BATCH padded_seqs.shape (from dataloader):", padded_seqs.shape)   # expect (B, S, F)
          #print("BATCH padding_mask.shape (constructed):", padding_mask.shape)     # expect (B, S)
          #print("padding_mask dtype:", padding_mask.dtype)
          optimizer.zero_grad()
          log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)  # ensure model expects this
          log_probs = log_probs.permute(1, 0, 2)
          # CTCLoss expects: (T, N, C) for inputs -> if model returns (T,N,C) ok, otherwise transpose
          # If model returns (N, T, C) -> do log_probs = log_probs.permute(1, 0, 2)
          # Ensure dtype: log_probs should be log probabilities (log_softmax)
          loss = ctc_loss(
              log_probs,
              padded_targets,
              seq_lengths,
              target_lengths
          )
          #print("Train log_probs shape (T,B,C):", log_probs.shape)
          #print("Train padded_targets shape:", padded_targets.shape)
          #print("seq_lengths:", seq_lengths[:5])
          #print("target_lengths:", target_lengths[:5])
          loss.backward()
          optimizer.step()

          epoch_loss += loss.item()
          if (batch_idx + 1) % 10 == 0:
              print(f"  Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

      avg_loss = epoch_loss / max(1, len(train_loader))
      print(f"End of Epoch {epoch+1}, Average Loss: {avg_loss:.4f} \n \n")

  # --- Save final checkpoint for this percentage (epoch is last epoch index) ---
  model_name = f"model_pct{percentage_str}_epoch{NUM_EPOCHS}.pth"
  checkpoint_path = os.path.join(CHECKPOINT_DIR, model_name)
  torch.save(model.state_dict(), checkpoint_path)
  print(f"Model checkpoint saved to {checkpoint_path}")
  print(f"Training Complete for percentage {percentage_str} % \n")
  return model, model_name, percentage_str, train_dataset, val_dataset, val_loader

SyntaxError: parameter without a default follows parameter with a default (ipython-input-4188074166.py, line 4)

In [ ]:
from torch.utils.data import DataLoader


TRAIN_DIR = '/content/drive/MyDrive/EEG-to-Text/preprocessed_frames_flat'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_CHANNELS = 127
NUM_EPOCHS = 30
BATCH_SIZE = 32
d_model = 256
num_encoder_layers = 4
dim_feedforward = 1024
CHECKPOINT_DIR = '/content/drive/MyDrive/EEG-to-Text/model_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
METRICS_CSV = os.path.join(CHECKPOINT_DIR, "evaluation_metrics.csv")

# Create CSV with header if missing
if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "percentage",
            "model_name",
            "epoch",
            "N_TRAIN",
            "N_VAL",
            "greedy_WER",
            "LM_WER",
            "LM_improvement"
        ])

print(f"Using device: {DEVICE}")

pct = 0.45
n_train = round(995 * pct)
n_val   = round(995 * (1-pct))
N_TRAIN = n_train
N_VAL = n_val
len_ds = 995
percentage_str = f"{round(N_TRAIN*100/995)}"
print(f"\n---------------- for {percentage_str}% : N_train = {N_TRAIN}, N_val = {N_VAL}")

dataset = EEGWordDataset_MultiSubject(
    data_dir=TRAIN_DIR,
    char_map=char_to_int,
    max_channels=MAX_CHANNELS,
    len_data=len_ds
)
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [N_TRAIN, len(dataset)-N_TRAIN])
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=0
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=0
)

sample_seq, _ = train_dataset[0]
HFA_FEATURE_DIM = sample_seq.shape[1]  # [SeqLen, FeatureDim]
print("HFA_FEATURE_DIM:", HFA_FEATURE_DIM)
model = BCIToTextModel(
    input_feat_dim=HFA_FEATURE_DIM,
    num_graphemes=NUM_GRAPHEMES,
    nhead=8,
    d_model=d_model,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward
).to(DEVICE)
ctc_loss = nn.CTCLoss(blank=BLANK_TOKEN_INDEX)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
model.train()
for epoch in range(NUM_EPOCHS):
  epoch_loss = 0.0
  for batch_idx, batch in enumerate(train_loader):
      padded_seqs, padded_targets, seq_lengths, target_lengths = [d.to(DEVICE) for d in batch]
      padded_seqs = padded_seqs[:, ::2, :]
      seq_lengths = (seq_lengths + 1) // 2

      max_seq_len = padded_seqs.shape[1]
      padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                      .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))
      max_seq_len = padded_seqs.shape[1]
      padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                      .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))
      optimizer.zero_grad()
      log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)
      log_probs = log_probs.permute(1, 0, 2)
      loss = ctc_loss(
          log_probs,
          padded_targets,
          seq_lengths,
          target_lengths
      )
      loss.backward()
      optimizer.step()
      epoch_loss += loss.item()
      if (batch_idx + 1) % 10 == 0:
          print(f"  Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

      avg_loss = epoch_loss / max(1, len(train_loader))
      print(f"End of Epoch {epoch+1}, Average Loss: {avg_loss:.4f} \n \n")

# --- Save final checkpoint for this percentage (epoch is last epoch index) ---
model_name = f"model_pct{percentage_str}_epoch{NUM_EPOCHS}.pth"
checkpoint_path = os.path.join(CHECKPOINT_DIR, model_name)
torch.save(model.state_dict(), checkpoint_path)
print(f"Model checkpoint saved to {checkpoint_path}")
print(f"Training Complete for percentage {percentage_str} % \n")


Using device: cpu

---------------- for 45% : N_train = 448, N_val = 547


NameError: name 'char_to_int' is not defined

In [ ]:
evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)

NameError: name 'model' is not defined

In [ ]:
import torch.nn.utils.rnn as rnn_utils
import csv
import time
import jiwer
import torch.optim as optim  # (even if unused)
def reconstruct_padded_targets_from_concat(targets_concat, target_lengths, pad_value):
    """
    Reconstruct a padded (B, max_U) target tensor from:
      - targets_concat: 1D tensor containing all targets concatenated
      - target_lengths: 1D tensor with each sample length

    Only for inspection/decoding (NOT for CTCLoss).
    """
    tgt_list = []
    offset = 0
    for L in target_lengths.cpu().tolist():
        seq = targets_concat[offset:offset + L].cpu()
        offset += L
        tgt_list.append(seq)

    padded_targets = rnn_utils.pad_sequence(
        tgt_list,
        batch_first=True,
        padding_value=pad_value
    )
    return padded_targets

def evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, N_TRAIN, N_VAL):
    # --- Evaluation ---
    print(f"Starting Model Evaluation for percentage {percentage_str} % \n")
    model.eval()
    model.to(DEVICE)

    all_targets = []
    all_greedy_preds = []
    all_lm_preds = []

    start_time = time.time()
    with torch.no_grad():
        for x, batch in enumerate(val_loader):
            # collate_fn now returns: (padded_seqs, targets_concat, seq_lengths, target_lengths)
            padded_seqs, targets_concat, seq_lengths, target_lengths = [
                d.to(DEVICE) for d in batch
            ]

            # Optional: shrink time dimension (T -> T/2)
            padded_seqs = padded_seqs[:, ::2, :]         # (B, T/2, F)
            seq_lengths = (seq_lengths + 1) // 2         # adjust lengths

            max_seq_len = padded_seqs.shape[1]
            padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                            .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))

            log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)  # (B, T, C)
            log_probs_for_decode = log_probs.permute(1, 0, 2).cpu()            # (T, B, C)

            #print("EVAL log_probs shape:", log_probs.shape)
            argmax_indices = log_probs.argmax(dim=2)  # (B, T)
            #print("Unique predicted classes:", argmax_indices.unique())
            #print("Counts per class:", torch.bincount(argmax_indices.view(-1)))

            #Reconstruct padded targets ONLY for decoding/printing
            padded_targets_for_print = reconstruct_padded_targets_from_concat(
                targets_concat, target_lengths, pad_value=BLANK_TOKEN_INDEX
            )

            # Changed from decode_target_batch to decode_padded_targets
            truth_text = decode_padded_targets(padded_targets_for_print.cpu(), target_lengths.cpu(), int_to_char)
            print("truth_text:", truth_text)

            greedy_text = greedy_decode_batch(
                log_probs_for_decode, int_to_char, BLANK_TOKEN_INDEX
            )
            print("greedy_text:", greedy_text)

            lm_text = beam_search_decode_batch(log_probs_for_decode, decoder)
            print("beam_search_decode_batch text:", lm_text)

            all_targets.extend(truth_text)
            all_greedy_preds.extend(greedy_text)
            all_lm_preds.extend(lm_text)

            if (x + 1) % 10 == 0:
                print(f"  Processed batch {x+1} / {len(val_loader)}")

    end_time = time.time()
    print(f"Evaluation Complete ({end_time - start_time:.2f}s)")

    greedy_wer = jiwer.wer(all_targets, all_greedy_preds)
    lm_wer = jiwer.wer(all_targets, all_lm_preds)
    improvement = greedy_wer - lm_wer

    print("\nPerformance Results ")
    print(f"  Greedy Decoding (No LM) WER:     {greedy_wer * 100:.2f}%")
    print(f"  Linguistic Refinement (LM) WER:  {lm_wer * 100:.2f}%")
    print("-" * 30)
    print(f"  Improvement from LM:             {improvement * 100:.2f}%")

    print("\nExample Decodings")
    print("TARGET".ljust(15), "| GREEDY".ljust(15), "| LM REFINED")
    print("-" * 45)
    for z in range(min(20, len(all_targets))):
        print(f"{all_targets[z]: <15} | {all_greedy_preds[z]: <15} | {all_lm_preds[z]}")
    print(f"Evaluation complete for {percentage_str}")

    # --- Save metrics to CSV immediately for this percentage ---
    with open(METRICS_CSV, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            percentage_str,
            model_name,
            NUM_EPOCHS,
            N_TRAIN,
            N_VAL,
            greedy_wer,
            lm_wer,
            improvement
        ])
    print(f"Saved metrics for {percentage_str}%  {METRICS_CSV}")


In [ ]:
Thus is not the version i aam using
import csv
import time
import jiwer
import torch.optim as optim

def evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, N_TRAIN, N_VAL):
  # --- Evaluation ---
  print(f"Starting Model Evaluation for percentage {percentage_str} % \n")
  model.eval()
  model.to(DEVICE)

  all_targets = []
  all_greedy_preds = []
  all_lm_preds = []

  start_time = time.time()
  with torch.no_grad():
      for x, batch in enumerate(val_loader):
          padded_seqs, padded_targets, seq_lengths, target_lengths = [d.to(DEVICE) for d in batch]

          padded_seqs = padded_seqs[:, ::2, :]          # (B, T/2, F)
          seq_lengths = (seq_lengths + 1) // 2          # adjust lengths
          max_seq_len = padded_seqs.shape[1]
          padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                          .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))
          log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)
          log_probs_for_decode = log_probs.permute(1, 0, 2).cpu()
          # print("During eval log_probs: ", log_probs_for_decode)
          # print("During eval log_probs shape:", log_probs.shape)
          # log_probs: (B, T, C)
          print("EVAL log_probs shape:", log_probs.shape)
          with torch.no_grad():
              argmax_indices = log_probs.argmax(dim=2)  # (B, T)

          print("Unique predicted classes:", argmax_indices.unique())
          print("Counts per class:", torch.bincount(argmax_indices.view(-1)))

          truth_text = decode_target_batch(padded_targets.cpu(), int_to_char)
          print("truth_text:" , truth_text)
          greedy_text = greedy_decode_batch(log_probs_for_decode, int_to_char, BLANK_TOKEN_INDEX)
          print("greedy_text:" , greedy_text)
          lm_text = beam_search_decode_batch(log_probs_for_decode, decoder)
          print("beam_search_decode_batch text:" , lm_text)

          all_targets.extend(truth_text)
          all_greedy_preds.extend(greedy_text)
          all_lm_preds.extend(lm_text)

          if (x + 1) % 10 == 0:
              print(f"  Processed batch {x+1} / {len(val_loader)}")

  end_time = time.time()
  print(f"Evaluation Complete ({end_time - start_time:.2f}s)")

  greedy_wer = jiwer.wer(all_targets, all_greedy_preds)
  lm_wer = jiwer.wer(all_targets, all_lm_preds)
  improvement = greedy_wer - lm_wer

  print("\nPerformance Results ")
  print(f"  Greedy Decoding (No LM) WER:     {greedy_wer * 100:.2f}%")
  print(f"  Linguistic Refinement (LM) WER:  {lm_wer * 100:.2f}%")
  print("-" * 30)
  print(f"  Improvement from LM:             {improvement * 100:.2f}%")

  print("\nExample Decodings")
  print("TARGET".ljust(15), "| GREEDY".ljust(15), "| LM REFINED")
  print("-" * 45)
  for z in range(min(20, len(all_targets))):
      print(f"{all_targets[z]: <15} | {all_greedy_preds[z]: <15} | {all_lm_preds[z]}")
  print(f"Evaluation complete for {percentage_str}")

  # --- Save metrics to CSV immediately for this percentage ---
  with open(METRICS_CSV, 'a', newline='') as f:
      writer = csv.writer(f)
      writer.writerow([
          percentage_str,
          model_name,
          NUM_EPOCHS,
          N_TRAIN,
          N_VAL,
          greedy_wer,
          lm_wer,
          improvement
      ])
  print(f"Saved metrics for {percentage_str}% → {METRICS_CSV}")

In [ ]:
# --- Paths & params ---
PROCESSED_DIR = '/content/drive/MyDrive/EEG-to-Text/preprocessed_frames'
TRAIN_DIR = '/content/drive/MyDrive/EEG-to-Text/preprocessed_frames_flat'
#TRAIN_DIR = os.path.join(PROCESSED_DIR, 'train')
VAL_DIR = os.path.join(PROCESSED_DIR, 'val')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_CHANNELS = 127
NUM_EPOCHS = 30
BATCH_SIZE = 32
d_model = 256
num_encoder_layers = 4
dim_feedforward = 1024
CHECKPOINT_DIR = '/content/drive/MyDrive/EEG-to-Text/model_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
METRICS_CSV = os.path.join(CHECKPOINT_DIR, "evaluation_metrics.csv")

# Create CSV with header if missing
if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "percentage",
            "model_name",
            "epoch",
            "N_TRAIN",
            "N_VAL",
            "greedy_WER",
            "LM_WER",
            "LM_improvement"
        ])

print(f"Using device: {DEVICE}")
N_per = [0.1]
'''
N_per = [0.7]
#, 0.6, 0.7, 0.8, 0.9, 1]
for pct in N_per:
    n_train = round(995 * pct)
    n_val   = round(995 * (1-pct))
    model, model_name, percentage_str, train_dataset, val_dataset, val_loader = trainModel(n_train, n_val, d_model, num_encoder_layers, dim_feedforward)
    evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)'''


Using device: cpu


'\nN_per = [0.7]\n#, 0.6, 0.7, 0.8, 0.9, 1]\nfor pct in N_per:\n    n_train = round(995 * pct)\n    n_val   = round(995 * (1-pct))\n    model, model_name, percentage_str, train_dataset, val_dataset, val_loader = trainModel(n_train, n_val, d_model, num_encoder_layers, dim_feedforward)\n    evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)'

In [ ]:
# --- Paths & params ---
PROCESSED_DIR = '/content/drive/MyDrive/EEG-to-Text/preprocessed_frames'
TRAIN_DIR = '/content/drive/MyDrive/EEG-to-Text/preprocessed_frames_flat'
#TRAIN_DIR = os.path.join(PROCESSED_DIR, 'train')
VAL_DIR = os.path.join(PROCESSED_DIR, 'val')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_CHANNELS = 127
NUM_EPOCHS = 50
BATCH_SIZE = 32
d_model = 256
num_encoder_layers = 4
dim_feedforward = 1024
CHECKPOINT_DIR = '/content/drive/MyDrive/EEG-to-Text/model_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
METRICS_CSV = os.path.join(CHECKPOINT_DIR, "evaluation_metrics.csv")

# Create CSV with header if missing
if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "percentage",
            "model_name",
            "epoch",
            "N_TRAIN",
            "N_VAL",
            "greedy_WER",
            "LM_WER",
            "LM_improvement"
        ])

print(f"Using device: {DEVICE}")
'''
N_per = [0.3]
#, 0.6, 0.7, 0.8, 0.9, 1]
for pct in N_per:
    n_train = round(995 * pct)
    n_val   = round(995 * (1-pct))
    model, model_name, percentage_str, train_dataset, val_dataset, val_loader = trainModel(n_train, n_val, d_model, num_encoder_layers, dim_feedforward)
    evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)'''


Using device: cpu


'\nN_per = [0.3]\n#, 0.6, 0.7, 0.8, 0.9, 1]\nfor pct in N_per:\n    n_train = round(995 * pct)\n    n_val   = round(995 * (1-pct))\n    model, model_name, percentage_str, train_dataset, val_dataset, val_loader = trainModel(n_train, n_val, d_model, num_encoder_layers, dim_feedforward)\n    evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)'

In [ ]:
pct =0.2
n_train = round(696 * pct)
n_val   = round(200 * pct)
n_test  = round(150 * pct)
model, model_name, percentage_str, val_loader = trainModel(n_train, n_val, n_test, d_model, num_encoder_layers, dim_feedforward)
evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)



In [ ]:
pct = 1
n_train = round(696 * pct)
n_val   = round(200 * pct)
n_test  = round(150 * pct)
model, model_name, percentage_str, val_loader = trainModel(n_train, n_val, n_test, d_model, num_encoder_layers, dim_feedforward)

evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)

In [ ]:
'''train_dataset = EEGWordDataset_MultiSubject(
      data_dir=TRAIN_DIR,
      char_map=char_to_int,
      max_channels=MAX_CHANNELS,
      len_data=100
  )
train_loader = DataLoader(
      train_dataset,
      batch_size=BATCH_SIZE,
      shuffle=True,
      collate_fn=ctc_collate_fn,
      num_workers=0
  )'''
one_batch = next(iter(train_loader))
padded_seqs, targets_concat, seq_lengths, target_lengths = [d.to(DEVICE) for d in one_batch]

# same preprocessing as in training
padded_seqs = padded_seqs[:, ::2, :]
seq_lengths = (seq_lengths + 1) // 2

max_seq_len = padded_seqs.shape[1]
padding_mask = (torch.arange(max_seq_len, device=DEVICE)
                .expand(len(seq_lengths), max_seq_len) >= seq_lengths.unsqueeze(1))

model = BCIToTextModel(
    input_feat_dim=padded_seqs.shape[2],
    num_graphemes=NUM_GRAPHEMES,
    nhead=4,
    d_model=256,
    num_encoder_layers=3,
    dim_feedforward=1024,
).to(DEVICE)

ctc_loss = nn.CTCLoss(blank=BLANK_TOKEN_INDEX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for step in range(2000):
    optimizer.zero_grad()
    log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)  # (B,T,C)
    log_probs_tbc = log_probs.permute(1, 0, 2)                         # (T,B,C)
    loss = ctc_loss(log_probs_tbc, targets_concat, seq_lengths, target_lengths)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"[Overfit] step {step}, loss={loss.item():.4f}")

In [ ]:
BLANK = BLANK_TOKEN_INDEX  # 0

with torch.no_grad():
    log_probs = model(padded_seqs, src_key_padding_mask=padding_mask)  # (B, T, C)

    #penalize blank a bit for decoding
    log_probs_adj = log_probs.clone()
    log_probs_adj[..., BLANK] -= 8.0

    log_probs_for_decode = log_probs_adj.permute(1, 0, 2).cpu()  # (T, B, C)
    print("after penalty = 8: ", log_probs_for_decode)

greedy_text = greedy_decode_batch(log_probs_for_decode, int_to_char, BLANK)
lm_text = beam_search_decode_batch(log_probs_for_decode, decoder)
print("Targets:", truth_text)
print("Greedy with blank penalty:", greedy_text)
print("LM with blank penalty :", lm_text)

In [ ]:
pct = 0.5
n_train = round(696 * pct)
n_val   = round(200 * pct)
n_test  = round(150 * pct)
model, model_name, percentage_str, val_loader = trainModel(n_train, n_val, n_test, d_model, num_encoder_layers, dim_feedforward)

evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)

In [ ]:
def decode_target_batch(target_ids, int_to_char, pad_index=BLANK_TOKEN_INDEX):
    """
    Decode integer target IDs to strings.

    Accepts:
      - 1D tensor: (U,)
      - 2D tensor: (B, U)

    Returns:
      - list of strings, length B (or [single_string] for 1D input)
    """
    # Convert lists to tensor if needed
    if isinstance(target_ids, list):
        target_ids = torch.tensor(target_ids, dtype=torch.long)

    # If it's 1D, make it (1, U) for uniform handling
    if target_ids.dim() == 1:
        target_ids = target_ids.unsqueeze(0)  # (1, U)

    decoded_batch = []
    # Now target_ids is always (B, U)
    for seq in target_ids:
        # seq is 1D (U,)
        seq_list = seq.tolist()  # convert to Python ints
        chars = [
            int_to_char.get(int(idx), "?")
            for idx in seq_list
            if int(idx) != pad_index
        ]
        decoded_batch.append("".join(chars))
    return decoded_batch


In [ ]:
def decode_padded_targets(padded_targets, target_lengths, int_to_char, pad_index=BLANK_TOKEN_INDEX):
    """
    padded_targets: (B, max_U)  tensor of ints
    target_lengths: (B,)        lengths of each target sequence
    Returns: list of strings
    """
    decoded_batch = []
    B = padded_targets.size(0)

    for i in range(B):
        L = int(target_lengths[i].item())
        seq = padded_targets[i, :L]          # (L,)
        seq_list = seq.tolist()
        chars = [
            int_to_char.get(int(idx), "?")
            for idx in seq_list
            if int(idx) != pad_index
        ]
        decoded_batch.append("".join(chars))

    return decoded_batch


In [ ]:
print("targets_concat shape:", targets_concat.shape)
print("target_lengths:", target_lengths)
print("padded_targets_for_print shape:", padded_targets_for_print.shape)

# reconstruct padded targets JUST for printing
# here `targets_concat` is actually a padded target matrix (B, max_U)
padded_targets = targets_concat  # rename for sanity

truth_text = decode_padded_targets(
    padded_targets.cpu(),
    target_lengths.cpu(),
    int_to_char,
    pad_index=BLANK_TOKEN_INDEX
)
print("truth_text:", truth_text)
greedy_text = greedy_decode_batch(log_probs_for_decode, int_to_char, BLANK_TOKEN_INDEX)

print("Targets:", truth_text)
print("Greedy :", greedy_text)


In [ ]:
print("Any 0 in targets used as real label?",
      any((targets_concat[i, :target_lengths[i]] == BLANK_TOKEN_INDEX).any().item()
          for i in range(targets_concat.size(0))))


In [ ]:
print("blank token:", labels[BLANK_TOKEN_INDEX])

In [ ]:
ctc_loss = nn.CTCLoss(blank=BLANK_TOKEN_INDEX, zero_infinity=True)
print("CTC blank index:", ctc_loss.blank)

In [ ]:
evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)

In [ ]:
model_file_path = '/content/drive/MyDrive/EEG-to-Text/model_checkpoints/model_pct60_epoch10.pth'
ckpt = torch.load(model_file_path, map_location="cpu")
filename = os.path.basename(model_file_path)
model_name = filename.replace(".pth", "")
percentage_str = "60"
epoch_str = "10"
ckpt.keys()

In [ ]:
import re

def load_bci_model_from_checkpoint(model_file_path,
                                   nhead=8,
                                   num_encoder_layers=6,
                                   dim_feedforward=2048,
                                   dropout=0.1,
                                   device="cuda" if torch.cuda.is_available() else "cpu"):
    """
    Load BCIToTextModel from a plain state_dict checkpoint and infer:
      - input_feat_dim from input_projection.weight
      - d_model from input_projection.weight
      - num_graphemes from fc_out.weight
    """
    state_dict = torch.load(model_file_path, map_location="cpu")

    # infer dimensions
    w_in = state_dict["input_projection.weight"]   # (d_model, input_feat_dim)
    w_out = state_dict["fc_out.weight"]           # (num_graphemes, d_model)
    w_out.bias[0] = -2.0
    d_model = w_in.shape[0]
    input_feat_dim = w_in.shape[1]
    num_graphemes = w_out.shape[0]

    print(f"Inferred from checkpoint -> d_model={d_model}, input_feat_dim={input_feat_dim}, num_graphemes={num_graphemes}")

    # instantiate model
    model = BCIToTextModel(
        input_feat_dim=input_feat_dim,
        num_graphemes=num_graphemes,
        nhead=nhead,
        d_model=d_model,
        num_encoder_layers=num_encoder_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    )

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    # extract model_name, percentage, epoch from filename
    filename = os.path.basename(model_file_path)
    model_name = filename.replace(".pth", "")

    m = re.search(r"pct(\d+)_epoch(\d+)", filename)
    percentage_str = m.group(1) if m else None
    epoch_str = m.group(2) if m else None

    print(f"Loaded model '{model_name}' (pct={percentage_str}, epoch={epoch_str}) on device {device}")

    return model, model_name, percentage_str, epoch_str, input_feat_dim, num_graphemes

In [ ]:
model, model_name, percentage_str, epoch_str, input_feat_dim, num_graphemes = load_bci_model_from_checkpoint(model_file_path)
n_train = round(696 * 0.6)
n_val   = 149
val_dataset = EEGWordDataset_MultiSubject(
  data_dir=VAL_DIR,
  char_map=char_to_int,
  max_channels=MAX_CHANNELS,
  len_data=n_val
  )
val_loader = DataLoader(
  val_dataset,
  batch_size=BATCH_SIZE,
  shuffle=False,
  collate_fn=ctc_collate_fn,
  num_workers=4
  )
evaluateAndSaveMetrics(model, model_name, percentage_str, val_loader, n_train, n_val)